# StreamLoop – Ajustando el Modelo de Cancelación

Freelance AI Engineer · clasificador de churn · ajuste sistemático de hiperparámetros.

**Prioridad de negocio:** un churn no detectado (FN) cuesta más que una oferta de retención innecesaria (FP). Por eso optimizamos **recall** de la clase `Churn=Yes`.

**Flujo obligatorio:**
1. Limpieza mínima → split train/test **antes** de cualquier otra operación
2. `Pipeline` con preprocesado + clasificador → **baseline** (defaults)
3. `RandomizedSearchCV` (amplio) → `GridSearchCV` (fino), solo sobre train
4. Revisar `cv_results_` (media **y** std entre folds)
5. Evaluar el modelo final en test **una sola vez** (el test solo se usa aquí y en el baseline)


## 0. Imports y configuración

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)
RANDOM_STATE = 42
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

print("OK – imports listos")


OK – imports listos


## 1. Carga y limpieza mínima

Solo operaciones que no implican aprendizaje: quitar ID, tipar `TotalCharges`, codificar el target.
La imputación / escalado / one-hot van **dentro** del `Pipeline`.


In [2]:
df = pd.read_csv(DATA_URL)
print(f"Shape original: {df.shape}")
print(df["Churn"].value_counts(normalize=True).round(3))

df = df.drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

y = df["Churn"].map({"Yes": 1, "No": 0}).astype(int)
X = df.drop(columns=["Churn"])

print(f"Features: {X.shape[1]} | NaNs en TotalCharges: {X['TotalCharges'].isna().sum()}")
X.head()


Shape original: (7043, 21)
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64
Features: 19 | NaNs en TotalCharges: 11


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


## 2. Train / test split **antes** de cualquier otra operación

El conjunto de test **no** se usa en la búsqueda. Solo se evalúa dos veces: baseline y modelo final.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Churn rate train={y_train.mean():.3f} | test={y_test.mean():.3f}")


Train: (5634, 19) | Test: (1409, 19)
Churn rate train=0.265 | test=0.265


## 3. Pipeline (preprocesado + clasificador)

In [4]:
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CATEGORICAL_FEATURES = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]

def build_pipeline() -> Pipeline:
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ])

pipeline = build_pipeline()
pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3I

## 4. Baseline — hiperparámetros por defecto

Entrenamos con defaults y medimos en test **una vez**. Esta es la referencia a batir.


In [5]:
def evaluate(model, X_te, y_te, title="Modelo"):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "precision_churn": precision_score(y_te, y_pred, pos_label=1),
        "recall_churn": recall_score(y_te, y_pred, pos_label=1),
        "f1_churn": f1_score(y_te, y_pred, pos_label=1),
        "roc_auc": roc_auc_score(y_te, y_proba),
        "confusion_matrix": confusion_matrix(y_te, y_pred),
    }
    print(f"=== {title} ===")
    for k, v in metrics.items():
        if k == "confusion_matrix":
            print(f"{k}:\n{v}")
        else:
            print(f"{k}: {v:.4f}")
    print("\n" + classification_report(y_te, y_pred, target_names=["No", "Yes"]))
    return metrics

baseline = build_pipeline()
baseline.fit(X_train, y_train)
baseline_metrics = evaluate(baseline, X_test, y_test, "Baseline (defaults)")


=== Baseline (defaults) ===
accuracy: 0.7850
precision_churn: 0.6228
recall_churn: 0.4813
f1_churn: 0.5430
roc_auc: 0.8200
confusion_matrix:
[[926 109]
 [194 180]]

              precision    recall  f1-score   support

          No       0.83      0.89      0.86      1035
         Yes       0.62      0.48      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.78      1409



## 5. Métrica de scoring alineada al negocio

No usamos accuracy (default de sklearn). Usamos **`recall`** de la clase positiva porque StreamLoop prioriza no dejar escapar cancelaciones.


In [6]:
SCORING = "recall"  # maximiza detección de Churn=Yes (minimiza FN)
print("scoring =", SCORING)
print(
    "Justificación: FN (churn no detectado) cuesta más que FP "
    "(retención ofrecida sin necesidad)."
)


scoring = recall
Justificación: FN (churn no detectado) cuesta más que FP (retención ofrecida sin necesidad).


## 6. `RandomizedSearchCV` — exploración amplia y barata

`n_jobs=-1`, CV, `refit=True`. **Solo se ajusta con `X_train` / `y_train`.**


In [7]:
random_pipe = build_pipeline()
random_param_distributions = {
    "classifier__n_estimators": [100, 200, 300, 400],
    "classifier__max_depth": [None, 5, 10, 15, 20, 30],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4, 8],
    "classifier__max_features": ["sqrt", "log2", None],
    "classifier__class_weight": [None, "balanced", "balanced_subsample"],
    "classifier__bootstrap": [True, False],
}

random_search = RandomizedSearchCV(
    estimator=random_pipe,
    param_distributions=random_param_distributions,
    n_iter=25,
    scoring=SCORING,
    cv=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1,
)
random_search.fit(X_train, y_train)

print(f"Mejor recall CV (random): {random_search.best_score_:.4f}")
print("Mejores params (random):")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")


Fitting 5 folds for each of 25 candidates, totalling 125 fits


Mejor recall CV (random): 0.7732
Mejores params (random):
  classifier__n_estimators: 200
  classifier__min_samples_split: 10
  classifier__min_samples_leaf: 8
  classifier__max_features: log2
  classifier__max_depth: 10
  classifier__class_weight: balanced_subsample
  classifier__bootstrap: False


## 7. `GridSearchCV` — refinado en la zona prometedora

Acotamos vecinos enteros alrededor de los mejores valores del random search.


In [8]:
bp = random_search.best_params_

def neighbors_int(value, options, pad=1):
    if value is None:
        return [None, 10, 15, 20]
    ordered = sorted(options)
    idx = ordered.index(value) if value in ordered else 0
    lo = max(0, idx - pad)
    hi = min(len(ordered), idx + pad + 1)
    return ordered[lo:hi]

n_est = bp["classifier__n_estimators"]
depth = bp["classifier__max_depth"]
mss = bp["classifier__min_samples_split"]
msl = bp["classifier__min_samples_leaf"]

grid_param_grid = {
    "classifier__n_estimators": sorted(set(neighbors_int(n_est, [100, 200, 300, 400]) + [n_est])),
    "classifier__max_depth": (
        [None, 10, 15, 20] if depth is None
        else sorted(set(neighbors_int(depth, [5, 10, 15, 20, 30]) + [depth]))
    ),
    "classifier__min_samples_split": sorted(set(neighbors_int(mss, [2, 5, 10, 20]) + [mss])),
    "classifier__min_samples_leaf": sorted(set(neighbors_int(msl, [1, 2, 4, 8]) + [msl])),
    "classifier__max_features": [bp["classifier__max_features"]],
    "classifier__class_weight": [bp["classifier__class_weight"]],
    "classifier__bootstrap": [bp["classifier__bootstrap"]],
}

print("Grid acotado:")
for k, v in grid_param_grid.items():
    print(f"  {k}: {v}")

grid_search = GridSearchCV(
    estimator=build_pipeline(),
    param_grid=grid_param_grid,
    scoring=SCORING,
    cv=5,
    n_jobs=-1,
    refit=True,  # reentrena best_estimator_ automáticamente
    verbose=1,
)
grid_search.fit(X_train, y_train)

print(f"\nMejor recall CV (grid): {grid_search.best_score_:.4f}")
print("Mejores params (grid / rank 1):")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")


Grid acotado:
  classifier__n_estimators: [100, 200, 300]
  classifier__max_depth: [5, 10, 15]
  classifier__min_samples_split: [5, 10, 20]
  classifier__min_samples_leaf: [4, 8]
  classifier__max_features: ['log2']
  classifier__class_weight: ['balanced_subsample']
  classifier__bootstrap: [False]
Fitting 5 folds for each of 54 candidates, totalling 270 fits



Mejor recall CV (grid): 0.8040
Mejores params (grid / rank 1):
  classifier__bootstrap: False
  classifier__class_weight: balanced_subsample
  classifier__max_depth: 5
  classifier__max_features: log2
  classifier__min_samples_leaf: 4
  classifier__min_samples_split: 5
  classifier__n_estimators: 100


## 8. Análisis de `cv_results_` — media **y** estabilidad

No nos quedamos solo con el mayor promedio: entre los top candidatos preferimos menor `std_test_score` si la media no cae más de 0.01.


In [9]:
results = pd.DataFrame(grid_search.cv_results_)
cols = ["rank_test_score", "mean_test_score", "std_test_score", "params"]
top = (
    results[cols]
    .sort_values(["mean_test_score", "std_test_score"], ascending=[False, True])
    .head(8)
    .reset_index(drop=True)
)
top["cv_lower"] = top["mean_test_score"] - top["std_test_score"]
display(top)

best_mean = top.loc[0, "mean_test_score"]
pool = top[top["mean_test_score"] >= best_mean - 0.01]
chosen = pool.sort_values("std_test_score", ascending=True).iloc[0]

print("\n=== Selección final ===")
print(f"mean_test_score: {chosen['mean_test_score']:.4f}")
print(f"std_test_score:  {chosen['std_test_score']:.4f}")
print(f"rank_test_score: {chosen['rank_test_score']}")
print("params:")
for k, v in chosen["params"].items():
    print(f"  {k}: {v}")
print(
    "\nJustificación: entre candidatos con media casi igual al mejor, "
    "elegimos el de menor varianza entre folds (más estable)."
)


,rank_test_score,mean_test_score,std_test_score,params,cv_lower
0,1,0.804013,0.016736,"{'classifier__bootstrap': False, 'classifier__...",0.787278
1,2,0.801338,0.013142,"{'classifier__bootstrap': False, 'classifier__...",0.788196
2,2,0.801338,0.015914,"{'classifier__bootstrap': False, 'classifier__...",0.785424
3,2,0.801338,0.016054,"{'classifier__bootstrap': False, 'classifier__...",0.785284
4,2,0.801338,0.016054,"{'classifier__bootstrap': False, 'classifier__...",0.785284
5,6,0.800669,0.015046,"{'classifier__bootstrap': False, 'classifier__...",0.785622
6,7,0.800000,0.014408,"{'classifier__bootstrap': False, 'classifier__...",0.785592
7,7,0.800000,0.015885,"{'classifier__bootstrap': False, 'classifier__...",0.784115



=== Selección final ===
mean_test_score: 0.8013
std_test_score:  0.0131
rank_test_score: 2
params:
  classifier__bootstrap: False
  classifier__class_weight: balanced_subsample
  classifier__max_depth: 5
  classifier__max_features: log2
  classifier__min_samples_leaf: 4
  classifier__min_samples_split: 10
  classifier__n_estimators: 100

Justificación: entre candidatos con media casi igual al mejor, elegimos el de menor varianza entre folds (más estable).


## 9. Modelo final y evaluación en test (única vez tras el baseline)

Si la configuración estable coincide con `best_params_`, usamos `best_estimator_` (ya reentrenado por `refit=True`). Si no, ajustamos esa configuración una vez sobre el train completo.


In [10]:
if chosen["params"] == grid_search.best_params_:
    final_model = grid_search.best_estimator_
    print("Usando grid_search.best_estimator_ (refit=True)")
else:
    final_model = build_pipeline()
    final_model.set_params(**chosen["params"])
    final_model.fit(X_train, y_train)
    print("Usando configuración estable distinta del rank 1; fit único sobre train")

tuned_metrics = evaluate(final_model, X_test, y_test, "Modelo ajustado (final)")


Usando configuración estable distinta del rank 1; fit único sobre train


=== Modelo ajustado (final) ===
accuracy: 0.7317
precision_churn: 0.4966
recall_churn: 0.7781
f1_churn: 0.6062
roc_auc: 0.8370
confusion_matrix:
[[740 295]
 [ 83 291]]

              precision    recall  f1-score   support

          No       0.90      0.71      0.80      1035
         Yes       0.50      0.78      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.79      0.73      0.75      1409



## 10. Comparación baseline vs ajustado

In [11]:
comparison = pd.DataFrame({
    "baseline": {
        "recall_churn": baseline_metrics["recall_churn"],
        "precision_churn": baseline_metrics["precision_churn"],
        "f1_churn": baseline_metrics["f1_churn"],
        "accuracy": baseline_metrics["accuracy"],
        "roc_auc": baseline_metrics["roc_auc"],
        "FN": int(baseline_metrics["confusion_matrix"][1, 0]),
        "FP": int(baseline_metrics["confusion_matrix"][0, 1]),
    },
    "tuned": {
        "recall_churn": tuned_metrics["recall_churn"],
        "precision_churn": tuned_metrics["precision_churn"],
        "f1_churn": tuned_metrics["f1_churn"],
        "accuracy": tuned_metrics["accuracy"],
        "roc_auc": tuned_metrics["roc_auc"],
        "FN": int(tuned_metrics["confusion_matrix"][1, 0]),
        "FP": int(tuned_metrics["confusion_matrix"][0, 1]),
    },
})
comparison["delta"] = comparison["tuned"] - comparison["baseline"]
display(comparison.round(4))

print(
    "\nConclusión: el recall de churn sube de forma clara y bajan los FN. "
    "La precisión baja (más FP), trade-off alineado con la prioridad de StreamLoop. "
    "Ver tuning_report.md para la justificación completa."
)


,baseline,tuned,delta
recall_churn,0.4813,0.7781,0.2968
precision_churn,0.6228,0.4966,-0.1263
f1_churn,0.5430,0.6062,0.0633
accuracy,0.7850,0.7317,-0.0532
roc_auc,0.8200,0.8370,0.0169
FN,194.0000,83.0000,-111.0000
FP,109.0000,295.0000,186.0000



Conclusión: el recall de churn sube de forma clara y bajan los FN. La precisión baja (más FP), trade-off alineado con la prioridad de StreamLoop. Ver tuning_report.md para la justificación completa.
